# FTQC 中間レポート: [[5,1,3]] 自作デコーダ(A到達版)

このノートブックでは、[[5,1,3]] 符号について以下を実装・確認する。

- **B**: 15エントリの syndrome -> recovery 表を完成させ、単一 X/Y/Z 誤り15ケースを全て訂正できることを確認する。
- **A**: 符号容量モデルで物理誤り率 p を振り、非符号化 baseline と比較して break-even を求める。

CX 誤り抽出回路と FT 必要性の調査(S課題)は今回の実装対象外とし、B/A の再現性を優先する。


## 0. 環境とセットアップ

標的環境: Qiskit 2.4.2 / qiskit-aer 0.17.2 / STIM 1.16.0。バージョンを確認します。

In [ ]:
# Colab で Qiskit が未導入の場合だけ依存関係を入れる。
# ローカルで既に入っている場合は何もしない。
import importlib.util, subprocess, sys
missing = [m for m in ['qiskit', 'qiskit_aer', 'matplotlib'] if importlib.util.find_spec(m) is None]
if missing:
    print('installing:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'qiskit', 'qiskit-aer', 'matplotlib'])
else:
    print('dependencies already available')


In [ ]:
# 基本の import。numpy は syndrome 計算(交換関係)に、Qiskit は状態準備と回路に使う。
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Pauli, StabilizerState
from qiskit.synthesis import synth_circuit_from_stabilizers  # スタビライザから状態準備回路を合成する
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

import qiskit, qiskit_aer
print("qiskit", qiskit.__version__, "| aer", qiskit_aer.__version__)
try:
    import stim
    print("stim", stim.__version__, "(任意セクションで使用)")
except Exception as e:
    print("stim 未導入(任意セクションのみ影響):", e)

## 1. 配布物: [[5,1,3]] 符号の定義

[[5,1,3]] は **非 CSS の完全符号**(距離3、5物理で1論理)。スタビライザは下の g1-g4。
論理演算子は X_L = XXXXX, Z_L = ZZZZZ。

非 CSS なので X 訂正と Z 訂正が分離せず、Steane 系(CSS)の復号パターンは移植できません。
そのため15通りの weight-1 Pauli(X/Y/Z 全部)を **1枚の syndrome 表に畳む** 必要があります。


In [ ]:
# スタビライザ生成元(配布値)。各文字列は左から qubit 0,1,2,3,4 上の Pauli。
G  = ['XZZXI', 'IXZZX', 'XIXZZ', 'ZXIXZ']   # g1, g2, g3, g4
XL = 'XXXXX'   # 論理 X
ZL = 'ZZZZZ'   # 論理 Z

# 完全符号の性質: 4スタビライザ -> 2^4 = 16 個の syndrome が
# 「誤りなし(I)」+「15個の weight-1 Pauli」と過不足なく1対1に対応する(衝突なし)。
# つまり recovery 表が一意に決まる。これが「15ケースで曖昧さゼロ」の根拠。
print("g1..g4 =", G)
print("X_L =", XL, " Z_L =", ZL)

In [ ]:
# 論理状態の準備回路(完成形・配布)。
# |0_L> は g1..g4 と Z_L のすべてを +1 固有値にする状態。
# synth_circuit_from_stabilizers にその5つを渡すと、準備回路が機械的に得られる(導出不要)。
prep0 = synth_circuit_from_stabilizers([*G, ZL])   # |0_L> を作る回路

# |1_L> は |0_L> に論理 X(= 各 qubit に X)を掛けたもの。
prep1 = prep0.copy()
for i, p in enumerate(XL):
    if p == 'X':
        prep1.x(i)

print("prep0 depth:", prep0.depth(), "ops:", dict(prep0.count_ops()))

# --- 自己検査: |0_L> が本当に g1..g4 と Z_L に +1 で固定されるか確認する ---
# (エンコーダが正しいことを実機で確かめる。あなたの実験の土台なので必ず通ること。)
ss0 = StabilizerState(prep0)
for s in [*G, ZL]:
    ev = np.real_if_close(ss0.expectation_value(Pauli(s)))
    assert abs(ev - 1) < 1e-9, f"{s} の固有値が +1 でない: {ev}"
# |1_L> では Z_L が -1 になるはず(論理ビットが反転している)
ss1 = StabilizerState(prep1)
zl_on_1 = np.real_if_close(ss1.expectation_value(Pauli(ZL)))
assert abs(zl_on_1 + 1) < 1e-9, f"|1_L> の Z_L が -1 でない: {zl_on_1}"
print("自己検査 OK: |0_L> は g1..g4,Z_L に +1、|1_L> は Z_L = -1")

## 2. syndrome worked example(1ケースのみ: X1)

**判定規則**: 誤り E とスタビライザ g が
- **反可換** なら syndrome ビット = **1**(g を測ると −1 が出る = 誤りが検出される)
- **可換** なら syndrome ビット = **0**

各 weight-1 誤りについて、g1-g4 それぞれとの可換/反可換を並べた4ビットが syndrome です。

### 例: X1(qubit 0 への X 誤り)

qubit 0 の上で、各生成元が持つ Pauli を見ると:

| 生成元 | qubit0 の Pauli | X との関係 | ビット |
|---|---|---|---|
| g1 = XZZXI | X | 可換(X と X) | 0 |
| g2 = IXZZX | I | 可換(X と I) | 0 |
| g3 = XIXZZ | X | 可換(X と X) | 0 |
| g4 = ZXIXZ | Z | 反可換(X と Z) | 1 |

X 誤りは qubit 0 にしか作用しないので、他の qubit の文字は無視してよい
(同じ qubit 上でのみ可換/反可換が問題になる)。よって

**s(X1) = (0, 0, 0, 1)**

下のセルでこの手順を実機でも確認します。


In [ ]:
# 交換関係の基本プリミティブ(配布・そのまま使ってよい)。
# 反可換 -> 1(syndrome ビットが立つ)、可換 -> 0、を返す。
def commutes_bit(pauli_a, pauli_b):
    """pauli_a と pauli_b が反可換なら 1、可換なら 0 を返す(syndrome ビットの定義そのもの)。"""
    return int(not Pauli(pauli_a).commutes(Pauli(pauli_b)))

# worked example: X1 の syndrome を g1..g4 に対して1つずつ求める。
X1 = 'XIIII'  # qubit 0 への X 誤り
s_X1 = tuple(commutes_bit(X1, g) for g in G)
print("s(X1) =", s_X1, " (手計算と一致するはず: (0, 0, 0, 1))")

## 3. 【あなたが作る①】デコーダ: 15エントリの (syndrome -> recovery) 表

worked example の手順(各 g との可換/反可換を4ビットに並べる)を、**残り14ケース**
(X2-X5, Y1-Y5, Z1-Z5)にも適用して、15個すべての syndrome を求めてください。

- 15個の syndrome は互いに相異なり、どれも (0,0,0,0) ではありません(完全符号の性質)。
- recovery は「その誤りを打ち消す Pauli」です。Pauli は自分自身が逆元(X·X=I)なので、
  weight-1 誤り E の recovery は E そのものになります。
- 完成した表(辞書など)から、syndrome を受け取って recovery を返す関数 `decode` を書いてください。

下に出発点として15個の誤りラベルを用意しました。`recovery_table` と `decode` を埋めてください。


In [ ]:
# 出発点: 扱うべき15個の weight-1 誤り。
def weight1(i, P):
    s = ['I'] * 5
    s[i] = P
    return ''.join(s)

ERRORS = [weight1(i, P) for i in range(5) for P in 'XYZ']
print('15個の誤り:', ERRORS)

# [[5,1,3]] は完全符号なので、15個の非ゼロ syndrome が15個の weight-1 Pauli と1対1対応する。
recovery_table = {
    (0, 0, 0, 1): 'XIIII',
    (1, 0, 1, 1): 'YIIII',
    (1, 0, 1, 0): 'ZIIII',
    (1, 0, 0, 0): 'IXIII',
    (1, 1, 0, 1): 'IYIII',
    (0, 1, 0, 1): 'IZIII',
    (1, 1, 0, 0): 'IIXII',
    (1, 1, 1, 0): 'IIYII',
    (0, 0, 1, 0): 'IIZII',
    (0, 1, 1, 0): 'IIIXI',
    (1, 1, 1, 1): 'IIIYI',
    (1, 0, 0, 1): 'IIIZI',
    (0, 0, 1, 1): 'IIIIX',
    (0, 1, 1, 1): 'IIIIY',
    (0, 1, 0, 0): 'IIIIZ',
}

def syndrome_of(error):
    return tuple(commutes_bit(error, g) for g in G)

def decode(syndrome):
    """4ビット syndrome から recovery の Pauli 文字列を返す。"""
    syndrome = tuple(syndrome)
    if syndrome == (0, 0, 0, 0):
        return 'IIIII'
    if syndrome not in recovery_table:
        raise ValueError(f'unknown syndrome: {syndrome}')
    return recovery_table[syndrome]

# 表の自己検査: 非ゼロ15 syndromeが重複なく埋まること、表が計算値と一致すること。
syndromes = [syndrome_of(e) for e in ERRORS]
assert len(set(syndromes)) == 15
assert (0, 0, 0, 0) not in syndromes
assert set(syndromes) == set(recovery_table)
for e in ERRORS:
    assert decode(syndrome_of(e)) == e
assert decode((0, 0, 0, 0)) == 'IIIII'

print('decoder table OK')
for e in ERRORS:
    print(f'{e:5s} syndrome={syndrome_of(e)} recovery={decode(syndrome_of(e))}')


## 4. 【あなたが作る②】単一誤り訂正の実証(15ケース)

任意の1量子ビットに X / Y / Z を注入し、syndrome を求め、`decode` で recovery を引いて訂正し、
**論理状態が保持されること** を15ケースすべてで確認してください。

訂正の成否は **残差 residual = recovery · error** で判定できます(解析的に最軽量):
- residual が g1-g4 と可換、かつ X_L・Z_L とも可換 なら、論理状態は保たれた(訂正成功)。
- そうでなければ論理誤り(訂正失敗)。

下に Pauli の積 `pauli_mul`(配布)を用意しました。成否判定 `correction_ok` と15ケースのループは
あなたが書いてください。


In [ ]:
# Pauli 文字列どうしの積(位相は無視。残差が論理を壊すかの判定にはこれで十分)。
_MUL = {'XY':'Z','YX':'Z','XZ':'Y','ZX':'Y','YZ':'X','ZY':'X'}
def pauli_mul(a, b):
    """2つの Pauli 文字列の積を文字列で返す(位相は無視)。"""
    out = []
    for x, y in zip(a, b):
        if x == 'I':   out.append(y)
        elif y == 'I': out.append(x)
        elif x == y:   out.append('I')
        else:          out.append(_MUL[x + y])
    return ''.join(out)

def correction_ok(error, recovery):
    """recovery * error がスタビライザ測定にも論理演算子にも影響しなければ成功。"""
    residual = pauli_mul(recovery, error)
    checks = [*G, XL, ZL]
    return all(commutes_bit(residual, op) == 0 for op in checks)

# 15ケースの実証: 任意の1量子ビットに X/Y/Z を注入し、復号後に論理が保存されることを確認する。
single_error_results = []
for error in ERRORS:
    s = syndrome_of(error)
    recovery = decode(s)
    residual = pauli_mul(recovery, error)
    ok = correction_ok(error, recovery)
    single_error_results.append((error, s, recovery, residual, ok))

print('error  syndrome      recovery residual ok')
for error, s, recovery, residual, ok in single_error_results:
    print(f'{error:5s} {str(s):14s} {recovery:8s} {residual:8s} {ok}')

assert all(ok for *_, ok in single_error_results)
print('single-error correction OK: 15/15')


## 5. 2つの雑音モデル(配布・別関数で明示分離)

**混同しないこと。** 比較の baseline は常に「符号容量モデル」、加点C で足すのが「CX 誤り」です。

- `noise_codecapacity(p)`(加点B 用): 抽出はノイズレス。各物理量子ビットに per-qubit depolarizing を
  1回だけ与える。これが clean な おおよそ pの2乗 を生む。**「CX エラーレート」とは呼ばない**
  (B では CX に誤りを置かないため)。
- `noise_cx(p)`(加点C 用): syndrome 抽出を **実回路** にして、抽出に使う2量子ビットゲート
  (X 支持の cx と Z 支持の cz)に depolarizing を乗せる。

両モデルとも **測定はノイズレス**(readout error は入れない)。


In [ ]:
# --- 符号容量モデル(加点B): per-qubit depolarizing を1回サンプルする ---
# depolarizing: 各 qubit で確率 p のとき X/Y/Z を等確率で1つ適用、確率 1-p で I。
# 抽出はノイズレスなので、サンプルした Pauli 誤り E がそのまま「真の誤り」になる。
_rng = np.random.default_rng(20260705)
def noise_codecapacity(p):
    """符号容量 / per-qubit depolarizing。5物理に独立に Pauli 誤りを置き、誤り文字列を返す。"""
    out = ['I'] * 5
    for i in range(5):
        if _rng.random() < p:
            out[i] = _rng.choice(['X', 'Y', 'Z'])
    return ''.join(out)

# 動作確認(誤りのサンプルが出るだけ。決め打ちの数値ではない)
print("符号容量サンプル例(p=0.2):", [noise_codecapacity(0.2) for _ in range(5)])

In [ ]:
# --- CX 誤りモデル(加点C): 実回路の2量子ビットゲートに depolarizing を乗せる NoiseModel ---
def noise_cx(p):
    """抽出回路の2量子ビットゲート(cx, cz)に 2量子ビット depolarizing(p)を付与する。
    非 CSS の重み4スタビライザは X 支持を cx、Z 支持を cz で測るため、両方に雑音を乗せないと
    Z 支持ゲートが無雑音になり加点C の雑音を過小評価する。測定はノイズレス(readout error なし)。"""
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p, 2), ['cx', 'cz'])
    return nm

# 重要な落とし穴: transpiler が抽出回路を最適化すると cx/cz が消えて雑音の置き場がずれる。
# optimization_level=0(または barrier)で抽出回路の構造を保持すること(#11 で既知の系と同型)。
print("noise_cx(0.05) 構築OK:", noise_cx(0.05).noise_instructions)

## 6. 実験ハーネス(骨・配布)

p を振って、符号化版(あなたの `decode` を通す)と非符号化版の論理誤り率を集計します。
**デコーダと成否判定はあなたの実装(セル3-4)を使います。** 下の枠はそのまま再利用してよい部分です。

非符号化 baseline は「1物理量子ビットに同じ depolarizing p を与えたメモリ比較」とします
(論理操作を伴わない素直な比較)。距離3 -> 1誤り訂正 -> 論理失敗には2誤り必要 -> 論理誤り
おおよそ pの2乗、という対応がプロットに現れるはずです。


In [ ]:
from itertools import product

def logical_error_rate_codecapacity(p, shots=20000):
    """符号容量モデルでの符号化版・論理誤り率を Monte Carlo で推定する。"""
    fails = 0
    for _ in range(shots):
        e = noise_codecapacity(p)
        s = syndrome_of(e)
        r = decode(s)
        if not correction_ok(e, r):
            fails += 1
    return fails / shots

def logical_error_rate_codecapacity_exact(p):
    """5量子ビット上の全Pauli誤り 4^5=1024 通りを数え上げ、論理誤り率を厳密に計算する。"""
    fail_prob = 0.0
    for chars in product('IXYZ', repeat=5):
        e = ''.join(chars)
        w = sum(c != 'I' for c in e)
        prob = (1 - p) ** (5 - w) * (p / 3) ** w
        r = decode(syndrome_of(e))
        if not correction_ok(e, r):
            fail_prob += prob
    return fail_prob

def unencoded_error_rate(p):
    """非符号化(1物理)メモリ比較の baseline。"""
    return p

def estimate_break_even(p_list, encoded, unenc):
    """隣接点の線形補間で encoded = unencoded となる p を推定する。"""
    diff = np.array(encoded) - np.array(unenc)
    for i in range(len(p_list) - 1):
        if diff[i] == 0:
            return p_list[i]
        if diff[i] * diff[i + 1] < 0:
            x0, x1 = p_list[i], p_list[i + 1]
            y0, y1 = diff[i], diff[i + 1]
            return x0 - y0 * (x1 - x0) / (y1 - y0)
    return None

# A課題の実験。exact は高速で再現性があるため、本文の数値にはこちらを使う。
P_LIST = [0.02, 0.04, 0.06, 0.08, 0.10, 0.13, 0.16, 0.20, 0.25, 0.30]
encoded_exact = [logical_error_rate_codecapacity_exact(p) for p in P_LIST]
unenc = [unencoded_error_rate(p) for p in P_LIST]
p_be = estimate_break_even(P_LIST, encoded_exact, unenc)

print('p      encoded_exact   unencoded')
for p, enc, base in zip(P_LIST, encoded_exact, unenc):
    print(f'{p:0.2f}   {enc:0.6f}        {base:0.6f}')
print('break-even estimate =', p_be)


In [ ]:
import matplotlib.pyplot as plt

def plot_breakeven(p_list, encoded, unenc, p_be=None):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(p_list, unenc, 'o-', label='unencoded (~ p)')
    ax.plot(p_list, encoded, 's-', label='encoded [[5,1,3]]')
    ax.plot(p_list, p_list, 'k:', alpha=0.4, label='y = p')
    if p_be is not None:
        ax.axvline(p_be, color='tab:red', linestyle='--', alpha=0.7, label=f'break-even ~ {p_be:.3f}')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('physical error rate p')
    ax.set_ylabel('logical error rate')
    ax.set_title('code-capacity break-even ([[5,1,3]])')
    ax.legend(); ax.grid(True, which='both', alpha=0.3)
    return fig

fig = plot_breakeven(P_LIST, encoded_exact, unenc, p_be)
plt.show()


## 7. 今回扱わない範囲: CX誤り抽出回路とFT考察

到達段階Aを確実にするため、本ノートブックではCX誤り抽出回路(S課題)は実装しない。
今回の比較は、抽出をノイズレスとみなす符号容量モデルに限定する。


In [ ]:
print('S課題(CX誤り抽出回路)は今回の実装対象外です。')


## 8. STIMについて

STIM は大規模な Clifford/Pauli シミュレーションに有用だが、A到達版では使わない。


In [ ]:
print('STIM は今回使用しません。')


## 9. 提出チェックリスト

- 到達段階: **A**
- デコーダ15表: 完成
- 単一 X/Y/Z 誤り15ケース: 全て訂正成功
- 符号容量モデル: encoded と unencoded を比較
- break-even: プロットと数値推定を表示
- S課題(CX誤り抽出/FT考察): 今回は対象外
